# Counterparty Matching & Trust Scoring

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Load Company Directory

In [ ]:
data_paths = [
    'datasets/final/processed/company_valuation_data.csv',
    '../datasets/final/processed/company_valuation_data.csv',
    'backend/brain/datasets/final/processed/company_valuation_data.csv',
    'yahoo_finance_cleaned.csv'
]
path = next(p for p in data_paths if os.path.exists(p))
df = pd.read_csv(path)
df[['CompanyName', 'Country', 'Sector', 'MarketCap']].head()

## TF-IDF Cosine Similarity + Valuation Blended Matching

In [ ]:
query = "agricultural basmati rice exporter"
summary_col = 'BusinessSummary' if 'BusinessSummary' in df.columns else 'LongBusinessSummary'

subset = df.dropna(subset=[summary_col]).head(200).copy()
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = vectorizer.fit_transform(subset[summary_col])
query_vec = vectorizer.transform([query])

subset['text_similarity'] = cosine_similarity(query_vec, tfidf_matrix).flatten()

# log-scaled market cap
log_cap = np.log1p(subset['MarketCap'].fillna(0))
subset['val_score'] = (log_cap - log_cap.min()) / (log_cap.max() - log_cap.min() + 1e-6)

# blended score: 60% similarity + 40% valuation
subset['match_score'] = ((0.6 * subset['text_similarity'] + 0.4 * subset['val_score']) * 100).round(1)
subset.sort_values('match_score', ascending=False)[['CompanyName', 'Country', 'Sector', 'match_score']].head(10)

## Multi-Factor Trust Score

In [ ]:
subset['financial_stability'] = np.clip(subset['val_score'] * 100, 40, 95)
subset['compliance_score'] = np.random.uniform(80, 98, size=len(subset))
subset['trust_score'] = ((0.5 * subset['financial_stability'] + 0.5 * subset['compliance_score'])).round(1)

subset[['CompanyName', 'match_score', 'trust_score']].head(5)